<a href="https://colab.research.google.com/github/stra-uss/academic/blob/main/te-299/notebooks/te_299_sentinel1_amazon_deforestation_s4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import numpy as np
from google.colab import drive, auth
from google.colab import userdata

In [4]:
print(" Montando Google Drive...")
drive.mount("/content/drive")
print("Drive montado.")

 Montando Google Drive...
Mounted at /content/drive
Drive montado.


In [6]:
!ls /content/drive/MyDrive/shared-public/sar-inpe-deter/outputs-labrea

01_deter_historico_mensal.png	05_correlacao_features.png
02_sar_series_temporais.png	dataset_ml_features.csv
03_sar_deter_sincronizados.png	dataset_sar_deter_labrea_completo.csv
04_boxplots_features.png	dataset_sar_deter.parquet


### __Label adjusting__

In [18]:
# Load the dataset
file_path = '/content/drive/MyDrive/shared-public/sar-inpe-deter/outputs-labrea/dataset_sar_deter_labrea_completo.csv'
df = pd.read_csv(file_path)
# Convert the 'datetime' column to datetime objects
df['datetime'] = pd.to_datetime(df['datetime'])
print(df.shape)
df.head(10)

(377, 38)


,CR_dB,CR_dB_sd,RFDI,RFDI_sd,RVI,RVI_sd,VH,VH_VV_ratio,VH_VV_ratio_sd,VH_sd,...,roll5_VH,d_CR_dB,roll3_CR_dB,roll5_CR_dB,d_RVI,roll3_RVI,roll5_RVI,d_RFDI,roll3_RFDI,roll5_RFDI
0,-6.781222,1.249293,0.645490,0.083490,0.709021,0.166981,-13.672358,0.218690,0.064182,1.133964,...,-13.672358,NaN,-6.781222,-6.781222,NaN,0.709021,0.709021,NaN,0.645490,0.645490
1,-6.020805,1.254300,0.592165,0.093163,0.815669,0.186327,-14.272940,0.260632,0.076973,1.166314,...,-13.972649,0.760417,-6.401014,-6.401014,0.106648,0.762345,0.762345,-0.053324,0.618827,0.618827
2,-6.006597,1.255373,0.591100,0.093330,0.817800,0.186659,-14.407880,0.261498,0.077163,1.175997,...,-14.117726,0.014208,-6.269541,-6.269541,0.002131,0.780830,0.780830,-0.001066,0.609585,0.609585
3,-5.883102,1.253157,0.581926,0.094589,0.836148,0.189178,-14.416768,0.268991,0.079097,1.192536,...,-14.192487,0.123495,-5.970168,-6.172932,0.018348,0.823206,0.794660,-0.009174,0.588397,0.602670
4,-5.819193,1.250632,0.577139,0.095198,0.845721,0.190397,-14.451100,0.272932,0.080126,1.388436,...,-14.244209,0.063909,-5.902964,-6.102184,0.009574,0.833223,0.804872,-0.004787,0.583388,0.597564
5,-5.989548,1.273286,0.589616,0.094822,0.820767,0.189644,-14.554036,0.262832,0.078577,1.514033,...,-14.420545,-0.170355,-5.897281,-5.943849,-0.024954,0.834212,0.827221,0.012477,0.582894,0.586389
6,-6.126544,1.277746,0.599618,0.093426,0.800763,0.186851,-14.696835,0.254741,0.076404,1.590354,...,-14.505324,-0.136996,-5.978428,-5.964997,-0.020004,0.822417,0.824240,0.010002,0.588791,0.587880
7,-6.107124,1.264849,0.598362,0.092824,0.803275,0.185648,-14.856884,0.255680,0.076074,1.490858,...,-14.595125,0.019420,-6.074405,-5.985102,0.002512,0.808269,0.821335,-0.001256,0.595866,0.589333
8,-6.082858,1.266602,0.596572,0.093103,0.806856,0.186205,-14.330386,0.257113,0.076330,1.297718,...,-14.577848,0.024266,-6.105509,-6.025053,0.003581,0.803631,0.815477,-0.001790,0.598184,0.592262
9,-6.214314,1.279007,0.605946,0.092603,0.788108,0.185206,-14.329259,0.249690,0.075210,1.556371,...,-14.553480,-0.131456,-6.134765,-6.104078,-0.018747,0.799413,0.803954,0.009374,0.600293,0.598023


In [19]:
print("Unique values in label_texto:", df['label_texto'].unique())

# If label_texto = 'FLORESTA' (or 'floresta'), label_ft = 0, else 1
df['label_ft'] = df['label_texto'].apply(lambda x: 0 if str(x).strip().upper() == 'FLORESTA' else 1)

print(df[['label_texto', 'label_ft']].drop_duplicates())

output_file_path = '/content/drive/MyDrive/shared-public/sar-inpe-deter/outputs-labrea/dataset_sar_deter_labrea_com_label_ft.csv'
df.to_csv(output_file_path, index=False)
print(f"File saved to {output_file_path}")

Unique values in label_texto: ['FLORESTA' 'DESMATAMENTO_CR' 'DEGRADACAO_VEG' 'CICATRIZ_FOGO']
         label_texto  label_ft
0           FLORESTA         0
2    DESMATAMENTO_CR         1
56    DEGRADACAO_VEG         1
306    CICATRIZ_FOGO         1
File saved to /content/drive/MyDrive/shared-public/sar-inpe-deter/outputs-labrea/dataset_sar_deter_labrea_com_label_ft.csv


In [20]:
df.label_ft.value_counts()

,count
label_ft,
0,204
1,173


### __Data augmentation and Sample data__

In [22]:
import numpy as np

# Separate classes
df_0 = df[df['label_ft'] == 0].copy()
df_1 = df[df['label_ft'] == 1].copy()

# Target samples per class
target_per_class = 250000

# Function to augment data with controlled Gaussian noise for numerical columns
def augment_class_data(df_class, target_count):
    # Separate types of columns
    num_cols = df_class.select_dtypes(include=[np.number]).columns.drop(['label_ft', 'label', 'ano', 'mes', 'dia_do_ano', 'trimestre', 'deter_n_alertas', 'janela_dias'])

    # We will resample with replacement to get the required number of rows
    np.random.seed(42)
    indices = np.random.choice(df_class.index, size=target_count, replace=True)
    augmented_df = df_class.loc[indices].copy().reset_index(drop=True)

    # Add minor noise to numerical radar features to prevent exact duplicates while preserving distribution
    # Noise scale will be a small percentage (e.g., 2%) of the standard deviation of each feature
    for col in num_cols:
        std_val = df_class[col].std()
        if pd.isna(std_val) or std_val == 0:
            std_val = 1e-4
        noise = np.random.normal(0, 0.02 * std_val, size=target_count)
        # Handle potential NaNs by filling them with the median before adding noise or keeping them
        augmented_df[col] = augmented_df[col] + noise

    # Temporal rigor: handle datetime, years, months, days based on the selected row
    # To introduce continuous variation without breaking consistency:
    # Shift datetime slightly by a random number of hours/minutes (e.g., +/- 12 hours max)
    time_shifts = np.random.randint(-12*3600, 12*3600, size=target_count)
    augmented_df['datetime'] = augmented_df['datetime'] + pd.to_timedelta(time_shifts, unit='s')

    # Recalculate time components derived directly from datetime to maintain strict temporal harmony
    augmented_df['ano'] = augmented_df['datetime'].dt.year
    augmented_df['mes'] = augmented_df['datetime'].dt.month
    augmented_df['dia_do_ano'] = augmented_df['datetime'].dt.dayofyear
    augmented_df['trimestre'] = augmented_df['datetime'].dt.quarter

    return augmented_df

# Generate datasets for both classes
print("Augmenting class 0...")
df_0_aug = augment_class_data(df_0, target_per_class)

print("Augmenting class 1...")
df_1_aug = augment_class_data(df_1, target_per_class)

# Combine datasets
balanced_df = pd.concat([df_0_aug, df_1_aug], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

print("New balanced dataset shape:", balanced_df.shape)
print(balanced_df['label_ft'].value_counts())

Augmenting class 0...
Augmenting class 1...
New balanced dataset shape: (500000, 39)
label_ft
0    250000
1    250000
Name: count, dtype: int64


In [23]:
# Save to a new CSV file
output_balanced_path = '/content/drive/MyDrive/shared-public/sar-inpe-deter/outputs-labrea/dataset_sar_deter_labrea_augmented_500k.csv'
balanced_df.to_csv(output_balanced_path, index=False)